# 02. Kiểm tra ảnh: ảnh thiếu, hỏng, hộp sai, ảnh trùng

Cần chạy **sau** `01_phan_tich_va_lam_sach_nhan.ipynb` và trên máy có thư mục ảnh `VNLP/detection`.
Mất vài phút cho 37.297 ảnh.

Kiểm tra:
- ảnh không tồn tại hoặc không mở được;
- hộp biển nằm ngoài ảnh;
- ảnh **trùng hoàn toàn** (MD5) và **gần trùng** (dHash, Hamming nhỏ hơn hoặc bằng 3), đặc biệt khi hai ảnh
  gần trùng lại **khác nhãn** (nhãn mâu thuẫn) hoặc **khác tập** (rò rỉ).

In [ ]:
import sys, csv, json, hashlib, collections, os, shutil, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from alpr.paths import load_paths

PATHS = load_paths()                  # khai báo trong config.json (xem config.example.json)
VNLP = PATHS["vnlp_root"]             # dữ liệu gốc (ổ E:)
DATASETS = PATHS["datasets_dir"]      # dataset sinh ra (để cùng ổ với VNLP cho hard link)
EXTERNAL = PATHS["external_dir"]      # video, bộ dữ liệu ngoài
manifest = pd.read_csv(ROOT / "data/labels/vnlp_manifest.csv", dtype={"label": str, "province": str}).fillna("")
print(len(manifest), "dòng manifest")

In [ ]:
def dhash(img: Image.Image) -> int:
    """Hash 64 bit theo chênh lệch độ sáng; ảnh gần giống nhau có hash gần nhau."""
    g = np.asarray(img.convert("L").resize((9, 8), Image.BILINEAR), dtype=np.int16)
    return int("".join("1" if b else "0" for b in (g[:, 1:] > g[:, :-1]).flatten()), 2)

def hamming(a: int, b: int) -> int:
    return bin(a ^ b).count("1")

## 1. Đọc toàn bộ ảnh

In [ ]:
records, hashes, md5s = [], {}, collections.defaultdict(list)
for i, r in enumerate(manifest.itertuples()):
    p = VNLP / r.rel_path.lstrip("/")
    rec = dict(rel_path=r.rel_path, exists=p.exists(), readable=False, width=None, height=None,
               bbox_ok=False, md5="", dhash="", issue="")
    if not rec["exists"]:
        rec["issue"] = "missing"
    else:
        try:
            data = p.read_bytes()
            with Image.open(p) as im:
                im.load()
                rec["width"], rec["height"] = im.size
                h = dhash(im)
            rec.update(readable=True, md5=hashlib.md5(data).hexdigest(), dhash=f"{h:016x}")
            hashes[i] = h
            md5s[rec["md5"]].append(i)
            rec["bbox_ok"] = 0 <= r.x1 < r.x2 <= rec["width"] and 0 <= r.y1 < r.y2 <= rec["height"]
            if not rec["bbox_ok"]:
                rec["issue"] = "bbox_out_of_image"
        except Exception as e:
            rec["issue"] = f"unreadable: {e}"
    records.append(rec)
    if (i + 1) % 5000 == 0:
        print(f"  {i + 1}/{len(manifest)}")

check = pd.DataFrame(records)
print(check.issue.replace("", "ok").value_counts().to_string())
check.groupby(["width", "height"]).size().sort_values(ascending=False).head()

## 2. Tìm ảnh trùng và gần trùng

So sánh mọi cặp 37 nghìn ảnh thì quá chậm. Mẹo: chia hash 64 bit thành 4 đoạn 16 bit. Hai hash lệch nhau
không quá 3 bit thì chắc chắn trùng nhau ở ít nhất 1 đoạn, nên chỉ cần so các ảnh cùng đoạn.

In [ ]:
MAX_HAMMING = 3
bands = collections.defaultdict(list)
for i, h in hashes.items():
    for b in range(4):
        bands[(b, (h >> (16 * b)) & 0xFFFF)].append(i)
pairs = set()
for idx in bands.values():
    if 2 <= len(idx) <= 200:
        for x in range(len(idx)):
            for y in range(x + 1, len(idx)):
                if hamming(hashes[idx[x]], hashes[idx[y]]) <= MAX_HAMMING:
                    pairs.add((min(idx[x], idx[y]), max(idx[x], idx[y])))
for ids in md5s.values():
    pairs.update((ids[x], ids[y]) for x in range(len(ids)) for y in range(x + 1, len(ids)))

dups = pd.DataFrame([dict(a=manifest.rel_path[i], b=manifest.rel_path[j],
                          label_a=manifest.label[i], label_b=manifest.label[j],
                          split_a=manifest.split[i], split_b=manifest.split[j],
                          hamming=hamming(hashes[i], hashes[j]))
                     for i, j in sorted(pairs)])
if len(dups):
    dups["kind"] = np.where(dups.label_a == dups.label_b, "same_plate", "DIFFERENT_LABEL")
    dups["cross_split"] = dups.split_a != dups.split_b
    print(dups.groupby(["kind", "cross_split"]).size())
else:
    print("Không có cặp ảnh trùng")

In [ ]:
# Xem vài cặp gần trùng nhưng khác nhãn hoặc khác tập (nếu có)
def show_pairs(d, n=6):
    d = d.head(n)
    if d.empty:
        print("(không có)"); return
    fig, ax = plt.subplots(len(d), 2, figsize=(8, 3 * len(d)), squeeze=False)
    for k, r in enumerate(d.itertuples()):
        for j, (p, lab, sp) in enumerate([(r.a, r.label_a, r.split_a), (r.b, r.label_b, r.split_b)]):
            ax[k, j].imshow(Image.open(VNLP / p.lstrip("/"))); ax[k, j].axis("off")
            ax[k, j].set_title(f"{lab} [{sp}]", fontsize=9)
    plt.tight_layout(); plt.show()

if len(dups):
    show_pairs(dups[(dups.kind == "DIFFERENT_LABEL") | dups.cross_split])

## 3. Xử lý và lưu

- Ảnh thiếu, hỏng hoặc hộp sai được ghi vào `reports/image_check.csv`; notebook 03 tự bỏ qua.
- Cặp gần trùng **khác tập**: đưa ảnh về cùng tập với ảnh còn lại, cập nhật vào manifest.
- Cặp gần trùng **khác nhãn**: xem bằng mắt rồi thêm vào `data/labels/label_fixes.csv` và chạy lại notebook 01.

In [ ]:
moved = 0
if len(dups):
    split_of = dict(zip(manifest.rel_path, manifest.split))
    for r in dups[dups.cross_split].itertuples():
        # đưa ảnh b về tập của ảnh a; ưu tiên giữ ảnh trong train
        target = "train" if "train" in (r.split_a, r.split_b) else r.split_a
        for p in (r.a, r.b):
            if split_of[p] != target:
                split_of[p] = target; moved += 1
    manifest["split"] = manifest.rel_path.map(split_of)
    manifest.to_csv(ROOT / "data/labels/vnlp_manifest.csv", index=False, encoding="utf-8")
print("Số ảnh được chuyển tập do gần trùng:", moved)

(ROOT / "reports").mkdir(exist_ok=True)
check.to_csv(ROOT / "reports/image_check.csv", index=False, encoding="utf-8")
if len(dups):
    dups.to_csv(ROOT / "reports/duplicates.csv", index=False, encoding="utf-8")
summary = dict(checked=len(check), issues=check.issue.replace("", "ok").value_counts().to_dict(),
               duplicate_pairs=len(dups), moved_to_same_split=moved)
(ROOT / "reports/image_check_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
summary